# **Category Database Creation and Linking**
This notebook categorizes meeting items and news articles and creates category nodes in the Neo4j database.
The categories are assigned by an LLM based on item context and linked to the knowledge graph.

## **1. Load Environment and Imports**

In [1]:
import os
import json
import time
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Load environment variables
load_dotenv("../config/config.env")
load_dotenv("../config/secrets.env")

# Import category linker functions
from data_pipeline.category_linker import (
    fetch_meeting_items_from_db,
    create_category_batch_file,
    submit_category_batch,
    check_category_batch_status,
    retrieve_batch_output,
    parse_category_results,
    create_category_nodes,
    link_meeting_items_to_categories,
    link_news_and_courses_to_categories,
    get_categorization_stats
)

## **2. Connect to Neo4j Database**

In [2]:
# Connect to Neo4j
uri = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(username, password))

# Test connection
with driver.session() as session:
    result = session.run("RETURN 1")
    print("Connected to Neo4j successfully!")
    print(f"Neo4j URI: {uri}")

Connected to Neo4j successfully!
Neo4j URI: neo4j://127.0.0.1:7687


## **3. Load Categories and Meeting Items**

In [3]:
# Load categories.json
# Get the path from environment variable
categories_path = os.getenv("CATEGORIES_JSON_PATH")
print(f"Categories path: {categories_path}")

with open(categories_path, "r", encoding="utf-8") as f:
    categories_data = json.load(f)

print(f"Loaded categories from {categories_path}")
print(f"Main categories: {len(categories_data.get('categories', []))}")

# Fetch all meeting items from database
meeting_items = fetch_meeting_items_from_db(driver)

Categories path: ../data/llm/schema/categories.json
Loaded categories from ../data/llm/schema/categories.json
Main categories: 7
Fetched 100 meeting items from database


## **3b. Load News Articles and Courses**


In [4]:
# Fetch all news articles from database
news_articles = []
with driver.session() as session:
    result = session.run("""
        MATCH (n:News)
        RETURN n.link as item_id, n.title as title, n.content as content
    """)
    for record in result:
        news_articles.append({
            'item_id': record['item_id'],
            'title': record['title'],
            'context': record['content'] or ""
        })

print(f"Loaded {len(news_articles)} news articles from database")
if news_articles:
    print(f"First news article: {news_articles[0]['title'][:80]}...")

# Fetch all courses from database
courses = []
with driver.session() as session:
    result = session.run("""
        MATCH (c:Course)
        RETURN c.coursecode as item_id, c.title as title, c.description as content
    """)
    for record in result:
        courses.append({
            'item_id': record['item_id'],
            'title': record['title'],
            'context': record['content'] or ""
        })

print(f"Loaded {len(courses)} courses from database")
if courses:
    print(f"First course: {courses[0]['title'][:80]}...")

# Combine meeting items, news, and courses for categorization
all_items = meeting_items + news_articles + courses
print(f"\nTotal items to categorize: {len(all_items)} (meetings: {len(meeting_items)}, news: {len(news_articles)}, courses: {len(courses)})")

Loaded 33 news articles from database
First news article: Kom ihåg förbudet mot utomhushållning av fjäderfä 8.2–31.5.2026...
Loaded 135 courses from database
First course: Träning för daglediga och seniorer...

Total items to categorize: 268 (meetings: 100, news: 33, courses: 135)


## **4. Create Category Batch File**

In [5]:
# Load the categorization prompt
prompt_path = os.getenv("CATEGORY_EXTRACTION_PROMPT_PATH")
with open(prompt_path, "r", encoding="utf-8") as f:
    prompt = f.read()

print(f"Loaded prompt from {prompt_path}")

# Create batch file for categorization (includes both meeting items and news)
batch_file_path = os.getenv("CATEGORY_BATCH_FILE_PATH")
create_category_batch_file(all_items, categories_data, prompt, batch_file_path)

Loaded prompt from ../data/llm/prompts/category_extraction_prompt.txt
Overwriting batch file...


Creating batch tasks: 100%|██████████| 268/268 [00:00<00:00, 5709.70it/s]

Batch file created at ../data/temp/category_batch.jsonl with 268 tasks


## **5. Submit Batch Job to OpenAI**

In [6]:
# Submit batch job
batch_id = submit_category_batch(
    batch_file_path,
    os.getenv("CATEGORY_BATCH_INPUT_ID_SAVE_PATH")
)

print(f"\nBatch submitted with ID: {batch_id}")
print("You can check the status in the next cell while waiting for completion.")

Batch job submitted successfully
Batch ID: batch_69c3bc3a70e881909a586a254a0a5d19
Batch ID saved at: ../data/temp/category_batch_file_id.txt

Batch submitted with ID: batch_69c3bc3a70e881909a586a254a0a5d19
You can check the status in the next cell while waiting for completion.


## **6. Poll Batch Status**
> This may take a few minutes to hours depending on the number of meeting items.
> You can run this cell periodically to check the status.

In [7]:
import time

# If you have a saved batch ID, load it
# batch_id = "batch_..."  # Replace with your batch ID

print(f"Checking status for batch: {batch_id}")
output_id = None

while output_id is None:
    output_id = check_category_batch_status(batch_id)
    if output_id is None:
        print("Batch still processing... waiting 10 seconds before next check")
        time.sleep(10)

print(f"\nBatch completed! Output file ID: {output_id}")

Checking status for batch: batch_69c3bc3a70e881909a586a254a0a5d19
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before 

In [ ]:
#Input llm batch output id for testing purposes
#output_id = "file-9dkjPBKtEZuA9ZuHZ4Dwf3"

## **7. Retrieve and Parse Results**

In [6]:
# Retrieve batch output from OpenAI
output_jsonl = retrieve_batch_output(output_id)

if output_jsonl is None:
    print("Error retrieving batch output!")
else:
    print(f"Retrieved output with {len(output_jsonl.splitlines())} lines")

# Parse category results
category_results = parse_category_results(output_jsonl)

print(f"\nParsing complete!")
print(f"Successfully categorized items: {len(category_results)}")

# Show sample results
sample_count = 0
for item_id, categories in category_results.items():
    if sample_count < 3:
        print(f"\nItem {sample_count + 1}:")
        print(f"  Item ID: {item_id}")
        print(f"  Categories: {len(categories)}")
        for cat in categories[:2]:
            print(f"    - {cat.get('category', 'N/A')} > {cat.get('subcategory', 'N/A')} (confidence: {cat.get('confidence', 0.0):.2f})")
        sample_count += 1
    else:
        break

Retrieved output with 268 lines
Parsed 268 category results

Parsing complete!
Successfully categorized items: 268

Item 1:
  Item ID: 4:b1daf7ce-04ab-42d9-8935-a8cc66a0837d:22
  Categories: 3
    - Barn och unga > Grundläggande utbildning och gymnasium (confidence: 0.90)
    - Barn och unga > Grundläggande utbildning och gymnasium (confidence: 0.85)

Item 2:
  Item ID: 4:b1daf7ce-04ab-42d9-8935-a8cc66a0837d:23
  Categories: 2
    - Barn och unga > Småbarnspedagogik och förskola (confidence: 0.95)
    - Kommunens organisation > Styrdokument och ekonomi (confidence: 0.85)

Item 3:
  Item ID: 4:b1daf7ce-04ab-42d9-8935-a8cc66a0837d:25
  Categories: 3
    - Barn och unga > Småbarnspedagogik och förskola (confidence: 0.90)
    - Barn och unga > Grundläggande utbildning och gymnasium (confidence: 0.80)


## **8. Create Category Nodes in Database**

In [7]:
# Create category nodes in Neo4j
create_category_nodes(driver, categories_data)

print("\nCategory nodes created successfully!")

Deleted all existing category nodes and relationships
Creating category nodes...


Processing categories: 100%|██████████| 7/7 [00:00<00:00, 22.24it/s]

Category nodes created successfully

Category nodes created successfully!


## **9. Link Meeting Items to Categories**

In [8]:
# Link meeting items to categories
link_meeting_items_to_categories(driver, category_results)

print("\nMeeting items linked to categories successfully!")

Linking meeting items to categories...


Creating meeting item relationships: 100%|██████████| 436/436 [00:03<00:00, 124.85it/s]

Category linking for meeting items completed

Meeting items linked to categories successfully!


## **9b. Link News Articles and Courses to Categories**


In [9]:
# Link news articles and courses to categories
link_news_and_courses_to_categories(driver, category_results)

Linking news articles and courses to categories...


Creating news and course relationships: 100%|██████████| 872/872 [00:06<00:00, 143.53it/s]

Category linking for news articles and courses completed


## **10. View Categorization Statistics**

In [10]:
# Get and display statistics
stats = get_categorization_stats(driver)

print("\n=== Categorization Statistics ===")
print(f"Total Category nodes: {stats['total_categories']}")
print(f"Total Subcategory nodes: {stats['total_subcategories']}")
print(f"Total SubSubcategory nodes: {stats['total_sub_subcategories']}")
print(f"\nMeeting items with categories: {stats['items_with_categories']}")
print(f"Total category links: {stats['total_category_links']}")
print(f"Average categories per item: {stats['total_category_links'] / max(stats['items_with_categories'], 1):.2f}")


=== Categorization Statistics ===
Total Category nodes: 7
Total Subcategory nodes: 30
Total SubSubcategory nodes: 0

Meeting items with categories: 205
Total category links: 325
Average categories per item: 1.59


## **11. Test Category Queries**
> Try some Cypher queries to verify the category structure and links

In [11]:
# Example query: Get all meeting items in a specific category
category_name = "Fritid och kultur"

with driver.session() as session:
    result = session.run("""
        MATCH (c:Category {name: $category_name})-[:HAS_SUBCATEGORY|HAS_SUB_SUBCATEGORY*0..]-(cat)
        <-[:HAS_CATEGORY]-(mi:MeetingItem)
        RETURN DISTINCT mi.title as title, count(*) as match_count
        LIMIT 10
    """, category_name=category_name)
    
    items = [record for record in result]
    print(f"\nMeeting items in '{category_name}':")
    for i, record in enumerate(items, 1):
        print(f"{i}. {record['title']}")
    
    if not items:
        print("No items found in this category")


Meeting items in 'Fritid och kultur':
1. Skapa-projektet Fem kommuner kring jämlik konstundervisning, intensionsavtal
2. Driftsbudget 2026 och ekonomiplan 2027-2028, medborgarinstitut
3. Medborgarinstitutets undervisningsplan för våren 2026
4. Arbetsperioder, läsåret 2025-2026, MI
5. MOTION, motion om utredning av för- och nackdelar med en sammanslagning av Malax-Korsnäs medborgarinstitut och Korsholms Vuxeninstitut
6. Delgivningsärenden, gemensamma skolväsendet och medborgarinstitutet
7. Anmälningsärenden, gemensamma skolväsendet och medborgarinstitutet
8. Medborgarinstitutets undervisningsplan för hösten 2025
9. Motion om användningen av kommunala utrymmen


In [14]:
# Example query: Find most common categories
with driver.session() as session:
    result = session.run("""
        MATCH (mi:MeetingItem)-[r:HAS_CATEGORY]->(cat)
        WHERE cat.name IS NOT NULL
        RETURN cat.name as category, avg(r.confidence) as avg_confidence, count(*) as count
        ORDER BY count DESC
        LIMIT 15
    """)
    
    categories = [record for record in result]
    print("\nMost common categories assigned:")
    for i, record in enumerate(categories, 1):
        print(f"{i}. {record['category']}: {record['count']} items (avg confidence: {record['avg_confidence']:.2f})")


Most common categories assigned:
1. Grundläggande utbildning och gymnasium: 49 items (avg confidence: 0.88)
2. Småbarnspedagogik och förskola: 26 items (avg confidence: 0.92)
3. Styrdokument och ekonomi: 22 items (avg confidence: 0.83)
4. Skolskjuts, morgon- och eftermiddagsverksamhet: 8 items (avg confidence: 0.94)
5. Bibliotek och medborgarinstitut: 8 items (avg confidence: 0.91)
6. Petalax-Nyby: 7 items (avg confidence: 0.83)
7. Elevvård och specialpedagogik: 6 items (avg confidence: 0.92)
8. Organisation och beslutsfattande: 6 items (avg confidence: 0.84)
9. Trafik och infrastruktur: 4 items (avg confidence: 0.67)
10. Övermalax: 3 items (avg confidence: 0.68)
11. Yttermalax: 3 items (avg confidence: 0.68)
12. Bergö: 3 items (avg confidence: 0.75)
13. Lediga jobb och praktik: 2 items (avg confidence: 0.85)
14. Kultur och kulturmiljö: 1 items (avg confidence: 0.95)
15. Påverkan och initiativ: 1 items (avg confidence: 0.75)


## **12. Close Database Connection**

In [15]:
# Close Neo4j connection
driver.close()
print("Database connection closed.")

Database connection closed.
